Ячейка 2 — Setup

In [1]:
# %%
"""
SETUP
"""
import json
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import pandas as pd

STRUCT = Path("../data/structured_data")
DICT_PATH = Path("../data/dictionary/result/dictionary_en_ru.json")
OUT_DIR = Path("../data/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DASH = "-"

print("STRUCT:", STRUCT.resolve())
print("DICT:", DICT_PATH.resolve())
print("OUT:", OUT_DIR.resolve())

STRUCT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/structured_data
DICT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json
OUT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output


Ячейка 3 — Dictionary

In [2]:
# %%
"""
DICTIONARY — guid → {en, ru}, en_lower → [ru, ...]
"""
with open(DICT_PATH, encoding="utf-8") as f:
    raw_dict = json.load(f)

guid_map = {}
en_lower_map = defaultdict(list)

for key, entry in raw_dict.items():
    if not isinstance(entry, dict):
        continue
    en = (entry.get("en") or "").strip()
    ru = (entry.get("ru") or "").strip()
    k = str(key).strip()
    guid_map[k.upper()] = entry
    guid_map[k.lower()] = entry
    if en:
        en_lower_map[en.lower()].append(ru if ru else None)

def translate_by_guid(guid: str | None) -> str:
    if not guid:
        return DASH
    e = guid_map.get(guid.upper()) or guid_map.get(guid.lower())
    if not e:
        return "(перевод не найден)"
    ru = (e.get("ru") or "").strip()
    if not ru:
        return "(перевод не найден)"
    return ru

def translate_by_en(text: str | None) -> str:
    if not text or text == DASH:
        return DASH
    variants = [v for v in en_lower_map.get(text.strip().lower(), []) if v]
    if not variants:
        return "(перевод не найден)"
    uniq = list(dict.fromkeys(variants))
    if len(uniq) > 1:
        return "(требуется ручная проверка)"
    return uniq[0]

print("guid entries:", len(guid_map) // 2)
print("en phrases:", len(en_lower_map))

guid entries: 116327
en phrases: 95870


Ячейка 4 — Load structured

In [3]:
# %%
"""
LOAD STRUCTURED
"""
def load_json(p: Path):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

cm_items = load_json(STRUCT / "classmods" / "all.json")
cm_passives = load_json(STRUCT / "classmods" / "passives_by_type.json")
cm_stats = load_json(STRUCT / "classmods" / "stats_by_type.json")
cm_legendaries = load_json(STRUCT / "classmods" / "legendaries.json")
cm_bodies = load_json(STRUCT / "classmods" / "bodies.json")

comps = load_json(STRUCT / "compositions" / "all.json")
bosses = load_json(STRUCT / "bosses" / "all.json")

# composition index by composition lower
comp_by_name = {}
for c in comps:
    nk = (c.get("composition") or "").lower()
    if nk:
        comp_by_name[nk] = c
    # also by internal
    inn = (c.get("internal_name") or "").lower()
    if inn and inn not in comp_by_name:
        comp_by_name[inn] = c

# bosses by boss_key
boss_by_key = {b["boss_key"]: b for b in bosses if b.get("boss_key")}

# pool_key → boss
pool_to_boss = {}
for b in bosses:
    for pk in b.get("pool_keys") or []:
        pool_to_boss[pk.lower()] = b

stat_g1 = (cm_stats or {}).get("stat_group1") or {}
stat_g2 = (cm_stats or {}).get("stat_group2") or {}

print("cm items:", len(cm_items))
print("comps:", len(comps))
print("bosses:", len(bosses))
print("passives types:", list(cm_passives.keys()))

cm items: 73
comps: 245
bosses: 82
passives types: ['classmod_dark_siren', 'classmod_exo_soldier', 'classmod_gravitar', 'classmod_paladin', 'classmod_robodealer']


Ячейка 5 — Microdicts + helpers

In [4]:
# %%
"""
MICRODICTS + HELPERS
"""
RARITY_RU = {
    "legendary": "Легендарный",
    "pearlescent": "Перламутровый",
    "pearl": "Перламутровый",
    "epic": "Фиолетовый",
    "rare": "Синий",
    "uncommon": "Зелёный",
    "common": "Белый",
}

CHARACTER_RU = {
    "dark_siren": "Тёмная сирена",
    "exo_soldier": "Экзо-солдат",
    "gravitar": "Гравитар",
    "paladin": "Паладин",
    "robodealer": "Рободилер",
}

# parser: только эти редкости
TARGET_RARITIES = {"legendary", "pearlescent", "pearl", "epic"}

def dash(x) -> str:
    if x is None:
        return DASH
    if isinstance(x, float) and pd.isna(x):
        return DASH
    s = str(x).strip()
    return s if s else DASH

def join_list(xs, sep="; ") -> str:
    if not xs:
        return DASH
    cleaned = [str(x).strip() for x in xs if x is not None and str(x).strip()]
    return sep.join(cleaned) if cleaned else DASH

def rarity_ru(r: str | None) -> str:
    if not r:
        return DASH
    return RARITY_RU.get(r.lower(), r)

def character_ru(ch: str | None) -> str:
    if not ch:
        return DASH
    return CHARACTER_RU.get(ch.lower(), ch)

def world_drop_ru(flag) -> str:
    if flag is True:
        return "есть"
    if flag is False:
        return "нет"
    return DASH

def flavor_text(block) -> tuple[str, str | None]:
    """→ (eng, guid)"""
    if not block or not isinstance(block, dict):
        return DASH, None
    text = block.get("text")
    guid = block.get("guid")
    if not text:
        return DASH, guid
    return str(text), guid

def origin_to_part(signals: list | None) -> str:
    if not signals:
        return "Base Game"
    # только то что есть в structured origin_signals
    return join_list(signals)

def resolve_sources(comp: dict | None) -> tuple[str, str, str]:
    """
    source (tech), source_name_eng, source_name_ru
    """
    if not comp:
        return DASH, DASH, DASH
    srcs = comp.get("drop_sources") or []
    tech_keys = []
    names_eng = []
    names_ru = []
    seen = set()

    for s in srcs:
        if isinstance(s, dict):
            key = str(s.get("key") or s.get("pool") or "").lower()
        else:
            key = str(s).lower()
        if not key or key in seen:
            continue
        seen.add(key)
        tech_keys.append(key)

        boss = pool_to_boss.get(key)
        if not boss:
            # itempoollist_arjay → arjay
            short = re.sub(r"^itempoollist_", "", key)
            short = re.sub(r"_(trueboss|true)$", "", short)
            boss = boss_by_key.get(short)

        if boss and boss.get("display_name"):
            names_eng.append(boss["display_name"])
            g = boss.get("display_guid")
            ru = translate_by_guid(g) if g else translate_by_en(boss["display_name"])
            names_ru.append(ru)
        else:
            names_eng.append(DASH)
            names_ru.append(DASH)

    return (
        join_list(tech_keys),
        join_list(names_eng) if any(x != DASH for x in names_eng) else DASH,
        join_list(names_ru) if any(x != DASH for x in names_ru) else DASH,
    )

Ячейка 6 — Build rows

In [5]:
# %%
"""
BUILD ROWS
Legendary / pearl — по composition;
Epic — по body (имя уникально, composition общий → drop/flavor часто -)
"""
rows = []

# быстрый index legendaries meta
leg_index = {}
for L in cm_legendaries:
    k = (L.get("item_type"), (L.get("composition") or "").lower())
    leg_index[k] = L

for item in cm_items:
    kind = item.get("kind") or ""
    rarity = (item.get("rarity") or "").lower() if item.get("rarity") else None
    item_type = item.get("item_type") or ""
    character = item.get("character")

    # фильтр редкостей
    if kind == "classmod_legendary":
        if rarity and rarity not in TARGET_RARITIES:
            # legendary/pearl always keep
            if rarity not in ("legendary", "pearlescent", "pearl"):
                continue
        if not rarity:
            rarity = "legendary"
    elif kind == "classmod_body":
        # body-строки = база имён для фиолетовых (и ниже);
        # в датасет кладём как Фиолетовый (пользователь просил epic+)
        rarity = "epic"
    else:
        continue

    composition = item.get("composition")
    comp = None
    if composition:
        comp = comp_by_name.get(composition.lower())

    # flavor / world / origin from composition when possible
    red_eng = red_guid = None
    leg_eng = leg_guid = None
    world = None
    part_eng = DASH

    if comp:
        red_eng, red_guid = flavor_text(comp.get("red_text"))
        leg_eng, leg_guid = flavor_text(comp.get("legendary_effect"))
        world = comp.get("has_world_drop")
        part_eng = origin_to_part(comp.get("origin_signals"))
        src_tech, src_eng, src_ru = resolve_sources(comp)
    else:
        red_eng, leg_eng = DASH, DASH
        src_tech, src_eng, src_ru = DASH, DASH, DASH

    # passives pool for class
    pas_list = cm_passives.get(item_type) or []
    pas_keys = []
    for p in pas_list:
        if isinstance(p, dict):
            pk = p.get("passive_key")
            if pk:
                pas_keys.append(pk)
        elif isinstance(p, str):
            pas_keys.append(p)
    # unique preserve order
    pas_keys = list(dict.fromkeys(pas_keys))

    # stats pools
    g1 = list(stat_g1.get(item_type) or [])
    g2 = list(stat_g2.get(item_type) or [])
    # for pure classmod shared pool also merge "classmod" key if present
    if not g1:
        g1 = list(stat_g1.get("classmod") or [])
    if not g2:
        g2 = list(stat_g2.get("classmod") or [])

    # legendary primary stats if any
    primary = item.get("primary_stats") or []
    if primary:
        # primary важнее для leg — можно показать в stat_group1 как fixed;
        # pool всё равно пишем полный
        pass

    name_eng = item.get("display_name") or DASH
    name_guid = item.get("display_guid")
    name_ru = translate_by_guid(name_guid) if name_guid else (
        translate_by_en(name_eng) if name_eng != DASH else DASH
    )

    rows.append({
        "item_id": item.get("item_id") or DASH,
        "name_eng": dash(name_eng),
        "name_ru": name_ru if name_eng != DASH else DASH,
        "rarity": rarity_ru(rarity),
        "character": character_ru(character),
        "red_text_eng": dash(red_eng),
        "red_text_ru": translate_by_guid(red_guid) if red_guid else (
            translate_by_en(red_eng) if red_eng not in (None, DASH) else DASH
        ),
        "legendary_effect_eng": dash(leg_eng),
        "legendary_effect_ru": translate_by_guid(leg_guid) if leg_guid else (
            translate_by_en(leg_eng) if leg_eng not in (None, DASH) else DASH
        ),
        "body": dash(item.get("body") or item.get("body_key")),
        "world_drop": world_drop_ru(world),
        "part_of_game_eng": dash(part_eng),
        "source": src_tech,
        "source_name_eng": src_eng,
        "source_name_ru": src_ru,
        "passive_points": join_list(pas_keys),
        "stat_group1": join_list(g1),
        "stat_group2": join_list(g2),
    })

print("rows:", len(rows))
print("by rarity:", pd.Series([r["rarity"] for r in rows]).value_counts().to_dict())
print("by character:", pd.Series([r["character"] for r in rows]).value_counts().to_dict())

rows: 73
by rarity: {'Фиолетовый': 50, 'Легендарный': 23}
by character: {'Тёмная сирена': 15, 'Экзо-солдат': 15, 'Гравитар': 15, 'Паладин': 15, 'Рободилер': 13}


Ячейка 7 — DataFrame + QA

In [6]:
# %%
"""
DATAFRAME + QA
"""
COLUMNS = [
    "item_id",
    "name_eng",
    "name_ru",
    "rarity",
    "character",
    "red_text_eng",
    "red_text_ru",
    "legendary_effect_eng",
    "legendary_effect_ru",
    "body",
    "world_drop",
    "part_of_game_eng",
    "source",
    "source_name_eng",
    "source_name_ru",
    "passive_points",
    "stat_group1",
    "stat_group2",
]

df = pd.DataFrame(rows, columns=COLUMNS)

# пустые → прочерк
for c in COLUMNS:
    df[c] = df[c].apply(lambda x: DASH if x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == "" else x)

print(df.shape)
print(df.head(10).to_string())
print("\nname filled:", (df["name_eng"] != DASH).sum(), "/", len(df))
print("source filled:", (df["source"] != DASH).sum(), "/", len(df))

(73, 18)
                                          item_id      name_eng           name_ru       rarity      character red_text_eng red_text_ru legendary_effect_eng legendary_effect_ru              body world_drop part_of_game_eng                                                                         source source_name_eng source_name_ru                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

Ячейка 8 — Save CSV

In [7]:
# %%
"""
SAVE CSV — bl4_classmods_MM-DD-YYYY_HH-MM-SS.csv
sep=';' — чтобы Excel (RU) сразу раскрыл столбцы
"""
ts = datetime.now().strftime("%m-%d-%Y_%H-%M-%S")
out_path = OUT_DIR / f"bl4_classmods_{ts}.csv"

# ; + utf-8-sig (BOM) — нормально открывается в Excel
df.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig",
    sep=";",
    quoting=1,  # csv.QUOTE_ALL — поля с ; и переносами не ломают таблицу
)

print("Written:", out_path.resolve())
print("rows:", len(df))
print("cols:", list(df.columns))
print(df.iloc[0].to_dict())

Written: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output/bl4_classmods_08-17-2026_16-23-17.csv
rows: 73
cols: ['item_id', 'name_eng', 'name_ru', 'rarity', 'character', 'red_text_eng', 'red_text_ru', 'legendary_effect_eng', 'legendary_effect_ru', 'body', 'world_drop', 'part_of_game_eng', 'source', 'source_name_eng', 'source_name_ru', 'passive_points', 'stat_group1', 'stat_group2']
{'item_id': 'classmod_dark_siren.comp_05_legendary_06', 'name_eng': 'Teen Witch', 'name_ru': 'ЦВЕТ ВЕДЬМОВСТВА', 'rarity': 'Легендарный', 'character': 'Тёмная сирена', 'red_text_eng': '-', 'red_text_ru': '-', 'legendary_effect_eng': '-', 'legendary_effect_ru': '-', 'body': 'leg_body_06', 'world_drop': 'есть', 'part_of_game_eng': 'Base Game', 'source': 'itempoollist_shatterlandsguardian; itempoollist_shatterlandsguardian_trueboss', 'source_name_eng': '-', 'source_name_ru': '-', 'passive_points': 'passive_blue_1_1_tier_1; passive_blue_1_1_tier_2; passive_blue_1_1_tier_3; passive_blue_1_1_tier_4; passive_blu